# Brain Tumor Segmentation & Radiomics Evaluation
This notebook is designed to run in Google Colab. It evaluates segmentation outputs against the BraTS 2020 dataset and performs radiomics feature extraction.

In [ ]:
# 1. Install required dependencies for Colab
!pip install kagglehub pyradiomics SimpleITK nibabel numpy scipy scikit-learn pandas

In [ ]:
import kagglehub
import os
import glob
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import pandas as pd
from radiomics import featureextractor
from google.colab import drive

In [ ]:
# 2. Download BraTS 2020 Dataset
path = kagglehub.dataset_download("awsaf49/brats20-dataset-training-validation")
print("Path to dataset files:", path)
brats_train_dir = os.path.join(path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")

In [ ]:
# 3. Unzip uploaded segmentation results
import zipfile
import os

zip_path = "/content/brats_pipeline_results (1).zip"
extract_dir = "/content/segmentation_outputs"

if os.path.exists(zip_path):
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Extraction complete.")

    # Print out the extracted directory structure to help debug
    print("\nExtracted files/folders:")
    for root, dirs, files in os.walk(extract_dir):
        level = root.replace(extract_dir, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files[:5]: # just show first 5 files per dir to not spam
            print(f"{subindent}{f}")
        if len(files) > 5:
            print(f"{subindent}... and {len(files)-5} more files")
else:
    print(f"Error: {zip_path} not found. Make sure you uploaded it to Colab.")

segmentation_dir = extract_dir


In [ ]:
# 4. Define Evaluation Metrics
def dice_coefficient(pred, gt):
    pred_bin = (pred > 0).astype(np.float32)
    gt_bin = (gt > 0).astype(np.float32)
    intersection = np.sum(pred_bin * gt_bin)
    denom = np.sum(pred_bin) + np.sum(gt_bin)
    if denom == 0:
        return 1.0 if np.sum(gt_bin) == 0 else 0.0
    return float(2.0 * intersection / denom)

def sensitivity(pred, gt):
    pred_bin = (pred > 0).astype(np.float32)
    gt_bin = (gt > 0).astype(np.float32)
    tp = np.sum(pred_bin * gt_bin)
    fn = np.sum((1 - pred_bin) * gt_bin)
    denom = tp + fn
    return float(tp / denom) if denom > 0 else float("nan")

def specificity(pred, gt):
    pred_bin = (pred > 0).astype(np.float32)
    gt_bin = (gt > 0).astype(np.float32)
    tn = np.sum((1 - pred_bin) * (1 - gt_bin))
    fp = np.sum(pred_bin * (1 - gt_bin))
    denom = tn + fp
    return float(tn / denom) if denom > 0 else float("nan")

def hausdorff_distance_95(pred, gt):
    from scipy.ndimage import distance_transform_edt, binary_erosion
    pred_bin = (pred > 0).astype(bool)
    gt_bin = (gt > 0).astype(bool)
    if not np.any(pred_bin) or not np.any(gt_bin):
        return float("nan")
    
    gt_dist = distance_transform_edt(~gt_bin)
    pred_surface = pred_bin & ~binary_erosion(pred_bin)
    distances_pred_to_gt = gt_dist[pred_surface]
    
    pred_dist = distance_transform_edt(~pred_bin)
    gt_surface = gt_bin & ~binary_erosion(gt_bin)
    distances_gt_to_pred = pred_dist[gt_surface]
    
    all_distances = np.concatenate([distances_pred_to_gt, distances_gt_to_pred])
    return float(np.percentile(all_distances, 95))

def compute_radiomics_icc(features_1, features_2):
    common_keys = set(features_1.keys()) & set(features_2.keys())
    icc_values = []
    for k in common_keys:
        v1, v2 = features_1[k], features_2[k]
        mean_v = (v1 + v2) / 2
        diff = abs(v1 - v2)
        if mean_v != 0:
            icc_proxy = max(0.0, 1.0 - diff / (abs(mean_v) * 2 + 1e-8))
        else:
            icc_proxy = 1.0 if diff < 1e-8 else 0.0
        icc_values.append(icc_proxy)
    return np.mean(icc_values) if icc_values else float('nan')

In [ ]:
# 5. Define Radiomics Extractor
def extract_radiomics(image_path, mask_path):
    # Setup standard pyradiomics settings
    settings = {'binWidth': 25, 'resampledPixelSpacing': None, 'interpolator': sitk.sitkBSpline}
    extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
    
    try:
        result = extractor.execute(image_path, mask_path)
        # Filter out diagnostic metadata keys
        return {k: float(v) for k, v in result.items() if not k.startswith("diagnostics_")}
    except Exception as e:
        print(f"Error extracting features: {e}")
        return None

In [ ]:
# 6. Run Full Evaluation Pipeline (With Batching & Auto-Save)
import glob
import os
import pandas as pd
import nibabel as nib
import numpy as np

# --- BATCHING SETTINGS ---
# Divide the 369 files into chunks (e.g., 0-90, 90-180, 180-270, 270-369)
BATCH_START = 0
BATCH_END = 95   # Change these numbers to process the next chunk!

all_files = sorted(glob.glob(os.path.join(segmentation_dir, "**", "*.nii.gz"), recursive=True))
segmentation_files = all_files[BATCH_START:BATCH_END]

print(f"Total files found in zip: {len(all_files)}")
print(f"Processing chunk from index {BATCH_START} to {BATCH_END} ({len(segmentation_files)} files)")

results = []
output_csv = f"/content/evaluation_results_batch_{BATCH_START}_to_{BATCH_END}.csv"

for seg_file in segmentation_files:
    basename = os.path.basename(seg_file)
    subject_id = basename.replace(".nii.gz", "").replace("_pred", "").replace("_seg", "")

    gt_path = os.path.join(brats_train_dir, subject_id, f"{subject_id}_seg.nii")
    t1ce_path = os.path.join(brats_train_dir, subject_id, f"{subject_id}_t1ce.nii")

    if not os.path.exists(gt_path):
        print(f"Ground truth not found for {subject_id}, skipping...")
        continue

    print(f"\nEvaluating {subject_id}...")

    # --- A. Segmentation Metrics ---
    pred_nib = nib.load(seg_file)
    gt_nib = nib.load(gt_path)

    pred_img = pred_nib.get_fdata()
    gt_img = gt_nib.get_fdata()

    # --- Shape Alignment Fix ---
    if pred_img.shape != gt_img.shape:
        if pred_img.shape == (155, 240, 240) and gt_img.shape == (240, 240, 155):
            pred_img = np.transpose(pred_img, (1, 2, 0))
            t1ce_nib = nib.load(t1ce_path)
            aligned_pred_nib = nib.Nifti1Image(pred_img, t1ce_nib.affine, t1ce_nib.header)
            temp_seg_file = "/content/temp_aligned_seg.nii.gz"
            nib.save(aligned_pred_nib, temp_seg_file)
            seg_file_for_radiomics = temp_seg_file
        else:
            print(f"    Warning: Shape mismatch! Pred: {pred_img.shape}, GT: {gt_img.shape}. Skipping...")
            continue
    else:
        seg_file_for_radiomics = seg_file

    d = dice_coefficient(pred_img, gt_img)
    sens = sensitivity(pred_img, gt_img)
    spec = specificity(pred_img, gt_img)
    hd95 = hausdorff_distance_95(pred_img, gt_img)

    print(f"  Segmentation -> Dice: {d:.4f}, Sens: {sens:.4f}, Spec: {spec:.4f}, HD95: {hd95:.4f}")

    # --- B. Radiomics Evaluation ---
    pred_features = {}
    gt_features = {}
    icc = float('nan')

    if os.path.exists(t1ce_path):
        print("  Extracting radiomics features...")
        pred_features = extract_radiomics(t1ce_path, seg_file_for_radiomics) or {}
        gt_features = extract_radiomics(t1ce_path, gt_path) or {}

        if pred_features and gt_features:
            icc = compute_radiomics_icc(pred_features, gt_features)
            print(f"  Radiomics    -> ICC: {icc:.4f}, Features extracted: {len(pred_features)}")
    else:
        print("  T1ce MRI not found, skipping radiomics...")

    # --- C. Build row with ALL data ---
    row = {
        'subject': subject_id,
        'dice': d,
        'sensitivity': sens,
        'specificity': spec,
        'hd95': hd95,
        'radiomics_icc': icc,
    }

    # Add all 107 individual radiomics features (from prediction mask)
    for feat_name, feat_val in pred_features.items():
        row[f"pred_{feat_name}"] = feat_val

    # Add all 107 individual radiomics features (from ground truth mask)
    for feat_name, feat_val in gt_features.items():
        row[f"gt_{feat_name}"] = feat_val

    results.append(row)

    # --- D. SAVE ON THE GO ---
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)

print("\n=========================================")
print(f"   BATCH {BATCH_START}-{BATCH_END} EVALUATION SUMMARY")
print("=========================================")
if results:
    summary_cols = ['dice', 'sensitivity', 'specificity', 'hd95', 'radiomics_icc']
    print(df[summary_cols].mean(numeric_only=True))
    print(f"\nTotal columns in CSV: {len(df.columns)} (6 metrics + {len(df.columns)-6} radiomics features)")
    print(f"Saved to: {output_csv}")
else:
    print("No results to display.")


In [ ]:
# 7. Export Results to CSV
if 'df' in locals() and not df.empty:
    output_csv = "/content/evaluation_results.csv"
    df.to_csv(output_csv, index=False)
    print(f"\nResults successfully saved to: {output_csv}")
    
    # Automatically trigger download to your local machine
    try:
        from google.colab import files
        files.download(output_csv)
    except Exception as e:
        print("Download trigger failed (are you running in Colab?). You can manually download it from the files tab.")
else:
    print("No results to save. Make sure you ran the evaluation cell above first!")
